In [64]:
import pandas as pd
import numpy as np
import json

In [65]:
TRAIN_SOURCES = ['doi:10.1200/PO.16.00054', 'doi:10.1158/2159-8290.CD-13-0617'] # 'Federica Catalanotti, David B. Solit', 'Eliezer M. Van Allen, Dirk Schadendorf' (45+66=111 patients) - full sequencing
VAL_SOURCES = ['doi:10.3390/cancers12082224', 'doi:10.3390/cancers11081203']    # 'Pauline Blateau, Jerome Solassol', 'Baptiste Louveau, Samia Mourah' (24+53=77 patients) - partial sequencing

## Clinical

In [66]:
def load_clinical_raw(split=None):
    if split == 'validation':
        df_patients = pd.read_csv("dataset/clinical.csv")
        df_patients = df_patients.loc[df_patients['source'].isin(VAL_SOURCES)]
    elif split == 'train':
        df_patients = pd.read_csv("dataset/clinical.csv")
        df_patients = df_patients.loc[df_patients['source'].isin(TRAIN_SOURCES)]

    else:
        raise ValueError("Please specify data split: 'train' or 'validation'")

    # drop irrelevant columns
    df_patients.drop(columns=['id', 'creation_datetime'], inplace=True)

    df_patients.replace(to_replace=['<NA>', 'nan', 'N.E.', None], value=np.nan, inplace=True)
    df_patients.reset_index(drop=True, inplace=True)
    return df_patients

load_clinical_raw(split='train')

,patientID,original_patientID,sex,age,AJCC_stage,M_stage,LDH,OS_status,OS_month,PFS_status,...,drug,BOR,BRAF_mut,brain_metastasis,immunotherapy_treatment,pre_MAPKi_treatment,CNA_data,SNV_data,GEX_data,source
0,CS_VT030,VT030,male,52,IV,M1C,normal,1.0,21.3,1.0,...,vemurafenib,PR,V600E,no,yes,Chemotherapy,yes,yes (bait_v4),no,doi:10.1200/PO.16.00054
1,CS_VT006,VT006,female,53,IIIC,NaN,elevated,0.0,46.6,0.0,...,dabrafenib,SD,V600E,no,yes,no,yes,yes (bait_v4),no,doi:10.1200/PO.16.00054
2,CS_VT014,VT014,male,63,IV,M1C,normal,0.0,23.7,0.0,...,vemurafenib,SD,V600E; R558Q,no,yes,no,yes,yes (bait_v3),no,doi:10.1200/PO.16.00054
3,CS_VT010,VT010,male,41,IV,M1C,NaN,1.0,23.3,1.0,...,vemurafenib,CR,V600E,yes,yes,no,yes,yes (bait_v3),no,doi:10.1200/PO.16.00054
4,CS_VT011,VT011,male,41,IV,M1C,elevated,0.0,41.8,1.0,...,vemurafenib,PR,V600E,no,yes,no,yes,yes (bait_v3),no,doi:10.1200/PO.16.00054
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,VS_Pat_66,Pat_66,male,71,IV,NaN,NaN,NaN,NaN,1.0,...,vemurafenib,PD,V600E,NaN,no,no,no,yes,no,doi:10.1158/2159-8290.CD-13-0617
107,VS_Pat_70,Pat_70,female,60,IV,NaN,NaN,NaN,NaN,1.0,...,vemurafenib,PR,V600E,NaN,no,no,no,yes,no,doi:10.1158/2159-8290.CD-13-0617
108,VS_Pat_73,Pat_73,male,51,IV,NaN,NaN,NaN,NaN,1.0,...,vemurafenib,PR,V600E,NaN,no,no,no,yes,no,doi:10.1158/2159-8290.CD-13-0617
109,VS_Pat_74,Pat_74,male,58,IV,NaN,NaN,NaN,NaN,1.0,...,vemurafenib,PR,V600E,NaN,no,no,no,yes,no,doi:10.1158/2159-8290.CD-13-0617


In [67]:
# Split drug categories
def split_drug_categories(split=None):
    """ Split drug categories to mono and combined drug therapy categories. """
    
    drug_pats = load_clinical_raw(split=split)[['patientID', 'drug']].copy()
    unique_drugs = ['vemurafenib', 'cobimetinib', 'dabrafenib', 'trametinib']
    drug_cats = dict()
    for _, (id, pat_drugs) in drug_pats.iterrows():
        if pat_drugs is np.nan:
            # pass
            drug_cats[id] = pd.Series([0 for _ in unique_drugs], index=unique_drugs)
        elif ' + ' in pat_drugs:
            drugs = pat_drugs.split(' + ')
            drug_cats[id] = pd.Series([1 for _ in drugs], index=drugs)
        else:
            drug_cats[id] = pd.Series([1], index=[pat_drugs])
    
    return pd.DataFrame(drug_cats).T.fillna(0)

def load_clinical_clean(split=None):
    
    df_patients = load_clinical_raw(split=split)
    
    # Drop brain_metastasis, immunotherapy_treatment, M_stage (many missing values)
    df_patients.drop(columns=['M_stage', 'brain_metastasis', 'immunotherapy_treatment', 'OS_status', 'OS_month'], inplace=True)
        
    # Remove missing values in PFS_status
    df_patients.dropna(subset=['PFS_status'], inplace=True) # (2 rows)
        
    # change str dtype events to numeric 
    df_patients['PFS_status'] = pd.to_numeric(df_patients.PFS_status)
    
    # remove patients with only post treatment snp data (6 rows)
    # df_patients = df_patients[df_patients.patientID.isin(['Pat_02', 'Pat_21', 'Pat_27', 'Pat_28', 'Pat_36', 'Pat_37', 'Pat_49']) == False]
    
    # drop feature AJCC_stage - high class imbalance
    df_patients.drop(columns=['AJCC_stage'], inplace=True)
    
    # drop few other class
    df_patients.drop(columns=['original_patientID', 'CNA_data', 'SNV_data', 'GEX_data'], inplace=True)

    # split drug categories
    drug_table = split_drug_categories(split=split)
    df_patients = df_patients.join(drug_table, on='patientID')
    df_patients.drop(columns=['drug'], inplace=True)
    
    # clean dataset
    df_patients.reset_index(drop=True, inplace=True)
    
    return df_patients

clinical_df = load_clinical_clean(split='train')


In [68]:
def count_unique_value_per_class(df):
    for feat in df.columns.tolist():
        print(f'{df[feat].value_counts(dropna=False)}\n')

count_unique_value_per_class(clinical_df)

patientID
CS_VT030     1
CS_VT006     1
CS_VT014     1
CS_VT010     1
CS_VT011     1
            ..
VS_Pat_66    1
VS_Pat_70    1
VS_Pat_73    1
VS_Pat_74    1
VS_Pat_76    1
Name: count, Length: 111, dtype: int64

sex
male      63
female    48
Name: count, dtype: int64

age
51    7
41    5
54    5
61    5
47    4
62    4
55    4
44    4
58    4
48    4
52    3
53    3
63    3
59    3
68    3
60    3
40    3
49    3
43    3
57    3
50    3
45    2
36    2
73    2
72    2
56    2
23    2
66    2
75    1
38    1
69    1
83    1
31    1
70    1
67    1
46    1
77    1
21    1
26    1
29    1
76    1
25    1
74    1
64    1
71    1
42    1
Name: count, dtype: int64

LDH
NaN         49
normal      38
elevated    24
Name: count, dtype: int64

PFS_status
1.0    95
0.0    16
Name: count, dtype: int64

PFS_month
3.0    5
2.2    5
1.5    4
5.2    4
3.5    3
      ..
8.2    1
8.8    1
2.8    1
7.2    1
7.0    1
Name: count, Length: 69, dtype: int64

BOR
PR     56
SD     25
PD     17
CR      9
NaN

## SNVS

In [69]:
def load_snps(split = None):
    if split == 'validation':
        df_snps = pd.read_csv("dataset/snvs.csv")
        df_snps = df_snps.loc[df_snps['source'].isin(VAL_SOURCES)]
    elif split == 'train':
        df_snps = pd.read_csv("dataset/snvs.csv")
        df_snps = df_snps.loc[df_snps['source'].isin(TRAIN_SOURCES)]
    else:
        raise ValueError("Please specify data split: 'train' or 'validation'")
    
    # get list of Catalanotti's sequenced genes
    bait = pd.ExcelFile("datafiles/catalanotti_supplement2.xlsx").parse(0)
    list_genes = list(bait['Gene Symbol'])
    df_snps = df_snps[df_snps.HGNC.isin(list_genes)]
    
    # drop irrelevant columns
    # df_snps.drop(columns=['id', 'creation_datetime', 'other_prelevements'], inplace=True)
    df_snps.drop(columns=['id', 'creation_datetime', 'HGVSp', 'other_prelevements', 'uniprot_id'], inplace=True)
    
    # patients with pre-treatment snp data
    df_snps = df_snps[df_snps.temporality == 'pre treatment']
    df_snps.drop(columns=['temporality'], inplace=True)
    
    # fill missing values
    snp_fills = {'consequence' : 'missing', 'variant_classification' : 'missing'}
    df_snps.fillna(value=snp_fills, inplace=True)
    df_snps.drop_duplicates(inplace=True)
    df_snps.reset_index(drop=True, inplace=True)
    return df_snps

snps_df = load_snps(split='train')
snps_df

,patientID,sample_id,HGNC,HGVSp_short,consequence,variant_classification,variant_type,chromosome,start_position,end_position,strand,ref_allele,tumor_allele_1,tumor_allele_2,source
0,CS_BW-09109-0376,DS-vem-065-T,BRAF,p.V600E,missense_variant,Missense_Mutation,SNP,7,140453136.0,140453136.0,+,A,A,T,doi:10.1200/PO.16.00054
1,CS_BW-09109-0376,DS-vem-065-T,KLF6,p.S233P,missense_variant,Missense_Mutation,SNP,10,3822401.0,3822401.0,+,A,A,G,doi:10.1200/PO.16.00054
2,CS_JJ-1446,DS-vem-066-T,EPHB2,p.E354K,missense_variant,Missense_Mutation,SNP,1,23191462.0,23191462.0,+,G,G,A,doi:10.1200/PO.16.00054
3,CS_JJ-1446,DS-vem-066-T,NTRK1,p.T705I,missense_variant,Missense_Mutation,SNP,1,156849858.0,156849858.0,+,C,C,T,doi:10.1200/PO.16.00054
4,CS_JJ-1446,DS-vem-066-T,CDC73,p.X473_splice,splice_acceptor_variant,Splice_Site,SNP,1,193218859.0,193218859.0,+,G,G,A,doi:10.1200/PO.16.00054
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1329,VS_Pat_41,Pat_41_Pre,FAS,p.C135Vfs*52,frameshift_variant,Frame_Shift_Del,DEL,10,90768708.0,90768708.0,+,T,T,-,doi:10.1158/2159-8290.CD-13-0617
1330,VS_Pat_06,Pat_06_Pre,DICER1,p.K1601Rfs*19,frameshift_variant,Frame_Shift_Del,DEL,14,95562455.0,95562455.0,+,T,T,-,doi:10.1158/2159-8290.CD-13-0617
1331,VS_Pat_41,Pat_41_Pre,CBL,p.N890Tfs*15,frameshift_variant,Frame_Shift_Del,DEL,11,119170435.0,119170435.0,+,A,A,-,doi:10.1158/2159-8290.CD-13-0617
1332,VS_Pat_53,Pat_53_Pre,FAS,p.C135Vfs*52,frameshift_variant,Frame_Shift_Del,DEL,10,90768708.0,90768708.0,+,T,T,-,doi:10.1158/2159-8290.CD-13-0617


In [70]:
count_unique_value_per_class(snps_df)

patientID
VS_Pat_76     52
CS_HS-0929    51
VS_Pat_24     42
CS_JJ-1446    39
CS_VT008T1    39
              ..
CS_JP-1351     2
CS_VT026       2
VS_Pat_55      2
VS_Pat_29      2
CS_JL-1415     1
Name: count, Length: 105, dtype: int64

sample_id
Pat_76_Pre      52
DS-vem-084-T    51
Pat_24_Pre      42
DS-vem-066-T    39
DS-vem-08-T1    39
                ..
DS-vem-082-T     2
DS-vem-26-T      2
Pat_55_Pre       2
Pat_29_Pre       2
DS-vem-085-T     1
Name: count, Length: 105, dtype: int64

HGNC
BRAF       106
GRIN2A      32
PTPRT       30
MLL3        24
PTPRD       24
          ... 
RAF1         1
CCNE1        1
RARA         1
BCL2L11      1
JUN          1
Name: count, Length: 235, dtype: int64

HGVSp_short
p.V600E           93
NaN                8
p.V600K            8
p.R58*             6
p.R201H            3
                  ..
p.K3870Rfs*19      1
p.K101Rfs*3        1
p.A2052Efs*215     1
p.K1601Rfs*19      1
p.N890Tfs*15       1
Name: count, Length: 1177, dtype: int64

consequenc

## KEGG, VCELLS

In [71]:
from scipy.stats import hypergeom
import os

############ CHECK REQUIRED FILES ############

# datafiles directory
DATAFILES = 'datafiles/'
# files with extracted pathways information
REQUIRED_FILES = [
    'Melanoma_vcells.csv',
    'Melanoma_kegg.csv',
    'Pathways_kegg.csv'
]

# files present in the directory
PRESENT_FILES = os.listdir(DATAFILES)
# check for required files in the directory
if len(np.intersect1d(REQUIRED_FILES, PRESENT_FILES)) < len(REQUIRED_FILES):
    print('Not enough required files')
    # print('Extracting data from resources.')
    # from extract_pathway_data import extract_data
    # extract_data()
    # print(f'Extracted data stored in "{DATAFILES}" directory')


In [72]:
############ MUTATION DATA ############

# function to get list of all mutated genes in patient from clinical and snps data
# used to integrate network/pathways data with patients data

def patients_mut_genes(split = None):
    """ Returns mapping of all mutated genes to respective patients. """
    snps_data = load_snps(split=split).copy()
    
    # if (split=='validation'): # Load mut genes for Blateau & Louveau patients
    #     snps_data = snps_data.loc[snps_data['source'].isin(VAL_SOURCES)]
    # elif split == 'train':
    #     # snps_data = snps_data[snps_data.source != 'doi:10.3390/cancers12082224'] # Pauline Blateau, Jerome Solassol
    #     snps_data = snps_data.loc[snps_data['source'].isin(TRAIN_SOURCES)]
    # else:
    #     raise ValueError("Please specify data split: 'train' or 'validation'")
    
    snps_data = snps_data[['patientID', 'HGNC']]

    return snps_data # cscape_genes_df

############ MELANOMA NETWORK ############

# create binary table of mutated genes respective to patients
def mut_genes_table(mutation_data, network_genes):
    """
    Function to create binary table of mutated genes in patient.

    arguments:
    df_snps : dataframe with patient id, HGNC symbols and mutation status
    network_genes : genes common between the patients and preferred pathway

    returns:
    pd.DataFrame : binary table of mutated genes

    """
    # get list of mutated genes for each patient
    mutated_df = mutation_data[mutation_data['HGNC'].isin(network_genes)].copy(deep=True)
    mutated_df.drop_duplicates(inplace=True)
    mutated_df = mutated_df.groupby(['patientID'], as_index=False).agg({'HGNC': lambda x: x.tolist()})
    # create binary table of mutated genes for patients
    mutation_dict = dict()
    for _, (id, genes) in mutated_df.iterrows():
        if id not in mutation_dict:
            mutation_dict[id] = pd.Series([1 for i in genes], index=genes)
    binary_table = pd.DataFrame(mutation_dict).T#.fillna(0)
    binary_table = pd.DataFrame(binary_table, columns=network_genes)
    binary_table = binary_table[sorted(binary_table.columns)]
    return binary_table


# map patients snp data to melanoma network from kegg
def map_melanoma_kegg(filename = None, split = None):
    mutation_data = patients_mut_genes(split=split).copy()

    if split=='validation':
        filename = 'Binary_kegg_validation.csv'
    elif split == 'train':
        filename = filename if filename is not None else 'Binary_kegg.csv'
    else:
        raise ValueError("Please specify data split: 'train' or 'validation'")

    if filename in PRESENT_FILES:
        return pd.read_csv(DATAFILES+filename, index_col=0)
    else:
        melanoma_kegg = pd.read_csv(DATAFILES+'Melanoma_kegg.csv')
        nodes_kegg = pd.concat([melanoma_kegg['src'], melanoma_kegg['dest']]).unique()  # unique values from src+dest (70 KEGG proteins) = ['AKT3', 'BRAF', etc]
        binary_kegg = mut_genes_table(mutation_data, nodes_kegg)    # like one hot encoding: patient - 70 proteins
        binary_kegg.to_csv(DATAFILES+filename, index=True)
        return binary_kegg
    
# map patients snp data to curated melanoma network from virtual cell
def map_melanoma_vcells(filename = None, split=None):
    mutation_data = patients_mut_genes(split=split).copy()

    if split=='validation':
        filename = 'Binary_vcells_validation.csv'
    elif split == 'train':
        filename = filename if not None else 'Binary_vcells.csv'
    else:
        raise ValueError("Please specify data split: 'train' or 'validation'")

    if filename in PRESENT_FILES:
        return pd.read_csv(DATAFILES+filename, index_col=0)
    else:
        melanoma_vcells = pd.read_csv(DATAFILES+'Melanoma_vcells.csv')
        nodes_vcells = pd.concat([melanoma_vcells['node1'], melanoma_vcells['node2']]).unique()
        binary_vcells = mut_genes_table(mutation_data, nodes_vcells)
        binary_vcells.to_csv(DATAFILES+filename, index=True)
        return binary_vcells
    
    
############ HYPERGEOMETRIC TEST ############

# Perform hypergeometric test of mutated genes in all pathways
def perform_hypergeometric_test(patient_mutations, pathway_mapping, genes_population):
    # population of protein coding genes
    M = len(genes_population)

    patients_pvals = {}  # store info
    for i, (patient, pat_genes) in patient_mutations.iterrows():
        # mutated genes in patient
        #pat_genes = df_snps_comp[df_snps_comp.patientID == patient].HGNC.unique()
        n = np.intersect1d(pat_genes, genes_population)
        
        pathway_pvals = {} # store pvals
        for pathway in pathway_mapping.pathway.unique():
            # total no. of genes in the pathway
            path_nodes = pathway_mapping[pathway_mapping.pathway == pathway].nodes.unique()
            N = np.intersect1d(path_nodes, genes_population)
            # no. of genes mutated in pathway
            #path_mut_genes = np.intersect1d(pat_genes, path_nodes)
            k = len(np.intersect1d(n, N))
            # perform hypergeometric test
            pval = hypergeom.pmf(k=k, M=M, n=len(n), N=len(N))
            value = -np.log10(pval)
            #lens = (len(pat_genes), len(n), len(path_nodes), len(N), len(path_mut_genes), len(k))
            #print(lens, pval, value)
            pathway_pvals[pathway] = value
        
        patients_pvals[patient] = pathway_pvals

    hg_results = pd.DataFrame(patients_pvals).T
    #print(hg_results.shape)
    return hg_results


# Calculate the pvalues for all pathways
def compute_pathway_pvalues(split=None):
    """ Perform hypergeometric test on mutated gene for all pathways and return pvalues """

    ## All protein coding genes fron HGNC database
    genes_population = pd.read_csv(DATAFILES+'hgnc_complete_set_2022-09-01.txt', sep='\t')
    genes_population = genes_population[genes_population.locus_group == 'protein-coding gene'].symbol
    #genes_population = genes_population[['symbol', 'alias_symbol', 'prev_symbol',]]

    ## Genes mapped to pathways
    # Load file with genes mapped to all pathways in kegg
    pathways_kegg = pd.read_csv(DATAFILES+'Pathways_kegg.csv')
    # Load file with genes mapped to Vcells melanoma pathway
    edges_vcells = pd.read_csv(DATAFILES+'Melanoma_vcells.csv')
    nodes_vcells = pd.concat([edges_vcells['node1'],edges_vcells['node2']]).unique()
    pathway_vcells = pd.DataFrame({'pathway': 'Melanoma Map (Curated)', 'nodes': nodes_vcells})
    # dataframe with genes mapped to all pathways (kegg + vcells)
    pathway_mapping = pd.concat([pathways_kegg, pathway_vcells], ignore_index=True)
    
    snps = load_snps(split=split)

    # # Patients mutation data
    # if split=='validation':
    #     # only patients with complete sequencing information
    #     snps = snps.loc[patient_mutations['source'].isin(VAL_SOURCES)]
    # elif split=='train':
    #     # only patients with complete sequencing information
    #     snps = snps[patient_mutations.source != 'doi:10.3390/cancers12082224'] # Pauline Blateau, Jerome Solassol
    # else:
    #     raise ValueError("Please specify data split: 'train' or 'validation'")
    
    patient_mutations = snps[['patientID', 'HGNC']].drop_duplicates()
    patient_mutations = patient_mutations.groupby(['patientID'], as_index=False).agg({'HGNC': lambda x: x.tolist()})

    ## Perform hypergeometric test for all pathways
    test_results = perform_hypergeometric_test(patient_mutations, pathway_mapping, genes_population)
    # test_results = perform_hypergeometric_test(cscape_genes_df, pathway_mapping, genes_population) # Cscape analysis

    ## Scale the values
    from sklearn.preprocessing import MinMaxScaler
    scaled_results = MinMaxScaler().fit_transform(test_results)
    scaled_results = pd.DataFrame(scaled_results, columns=test_results.columns, index=test_results.index)

    return scaled_results


# Load computed pathway pvalues
def get_pathway_pvalues(filename = None, split = None):
    
    if split=='validation':
        filename='Pathway_pvalues_validation.csv'
    elif split=='train':
        filename = filename if not None else 'Pathway_pvalues.csv'
    else:
        raise ValueError("Please specify data split: 'train' or 'validation'")
    
    if filename in PRESENT_FILES:
        return pd.read_csv(DATAFILES+filename, index_col=0)
    else:
        print('computing pvalues ....')
        pval_df = compute_pathway_pvalues(split=split)
        pval_df.to_csv(DATAFILES+filename, index=True)
        return pval_df


In [73]:
from scipy.stats import hypergeom
import os

############ CHECK REQUIRED FILES ############

# datafiles directory
DATAFILES = 'datafiles/'
# files with extracted pathways information
REQUIRED_FILES = [
    'Melanoma_vcells.csv',
    'Melanoma_kegg.csv',
    'Pathways_kegg.csv'
]

# files present in the directory
PRESENT_FILES = os.listdir(DATAFILES)
# check for required files in the directory
if len(np.intersect1d(REQUIRED_FILES, PRESENT_FILES)) < len(REQUIRED_FILES):
    print('Not enough required files')
    # print('Extracting data from resources.')
    # from extract_pathway_data import extract_data
    # extract_data()
    # print(f'Extracted data stored in "{DATAFILES}" directory')


In [74]:
full_genes="CAT"

if full_genes == 'ALL':
    # Melanoma network (KEGG)
    binary_kegg = map_melanoma_kegg(split='train')
    # Melanoma network (Virtual cell)
    binary_vcells = map_melanoma_vcells(split='train')
    # Pathway pvalues
    pathway_pvalues = get_pathway_pvalues(split='train')
elif full_genes == 'CAT':
    #### For Catalanotti Analysis (CAT)
    # Melanoma network (KEGG)
    binary_kegg = map_melanoma_kegg('CAT_Binary_kegg.csv', split='train')
    # Melanoma network (Virtual cell)
    binary_vcells = map_melanoma_vcells('CAT_Binary_vcells.csv', split='train')
    # Pathway pvalues
    pathway_pvalues = get_pathway_pvalues('CAT_Pathway_pvalues.csv', split='train')
elif full_genes == 'CA':
    # #### For Cscape Analysis (CA)
    # Melanoma network (KEGG)
    binary_kegg = map_melanoma_kegg('CA_Binary_kegg.csv', split='train')
    # Melanoma network (Virtual cell)
    binary_vcells = map_melanoma_vcells('CA_Binary_vcells.csv', split='train')
    # Pathway pvalues
    pathway_pvalues = get_pathway_pvalues('CA_Pathway_pvalues.csv', split='train')
    

In [75]:
clinical_data = load_clinical_clean(split='train')


In [76]:
########## MELANOMA NETWORK (KEGG) ##########

# join clinical and network data on patient id
clinical_kegg = clinical_data.join(binary_kegg, on='patientID')
clinical_kegg[binary_kegg.columns] = clinical_kegg[binary_kegg.columns].fillna(value=0)
clinical_kegg

,patientID,sex,age,LDH,PFS_status,PFS_month,BOR,BRAF_mut,pre_MAPKi_treatment,source,...,PIK3CB,PIK3CD,PIK3R1,PIK3R2,PIK3R3,POLK,PTEN,RAF1,RB1,TP53
0,CS_VT030,male,52,normal,1.0,6.5,PR,V600E,Chemotherapy,doi:10.1200/PO.16.00054,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0
1,CS_VT006,female,53,elevated,0.0,44.6,SD,V600E,no,doi:10.1200/PO.16.00054,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,CS_VT014,male,63,normal,0.0,6.4,SD,V600E; R558Q,no,doi:10.1200/PO.16.00054,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,CS_VT010,male,41,NaN,1.0,8.0,CR,V600E,no,doi:10.1200/PO.16.00054,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,CS_VT011,male,41,elevated,1.0,9.9,PR,V600E,no,doi:10.1200/PO.16.00054,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,VS_Pat_66,male,71,NaN,1.0,1.5,PD,V600E,no,doi:10.1158/2159-8290.CD-13-0617,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
107,VS_Pat_70,female,60,NaN,1.0,7.2,PR,V600E,no,doi:10.1158/2159-8290.CD-13-0617,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
108,VS_Pat_73,male,51,NaN,1.0,6.2,PR,V600E,no,doi:10.1158/2159-8290.CD-13-0617,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
109,VS_Pat_74,male,58,NaN,1.0,7.0,PR,V600E,no,doi:10.1158/2159-8290.CD-13-0617,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [77]:
count_unique_value_per_class(clinical_kegg)

patientID
CS_VT030     1
CS_VT006     1
CS_VT014     1
CS_VT010     1
CS_VT011     1
            ..
VS_Pat_66    1
VS_Pat_70    1
VS_Pat_73    1
VS_Pat_74    1
VS_Pat_76    1
Name: count, Length: 111, dtype: int64

sex
male      63
female    48
Name: count, dtype: int64

age
51    7
41    5
54    5
61    5
47    4
62    4
55    4
44    4
58    4
48    4
52    3
53    3
63    3
59    3
68    3
60    3
40    3
49    3
43    3
57    3
50    3
45    2
36    2
73    2
72    2
56    2
23    2
66    2
75    1
38    1
69    1
83    1
31    1
70    1
67    1
46    1
77    1
21    1
26    1
29    1
76    1
25    1
74    1
64    1
71    1
42    1
Name: count, dtype: int64

LDH
NaN         49
normal      38
elevated    24
Name: count, dtype: int64

PFS_status
1.0    95
0.0    16
Name: count, dtype: int64

PFS_month
3.0    5
2.2    5
1.5    4
5.2    4
3.5    3
      ..
8.2    1
8.8    1
2.8    1
7.2    1
7.0    1
Name: count, Length: 69, dtype: int64

BOR
PR     56
SD     25
PD     17
CR      9
NaN

In [78]:
########## MELANOMA NETWORK (VCELLS) ##########

# join clinical and network data on patient id
clinical_vcells = clinical_data.join(binary_vcells, on='patientID')
# # Drop empty columns
# #clinical_vcells.dropna(how='all', axis=1, inplace=True)
# drop_vcells = threshold_level(clinical_vcells, binary_vcells.columns)
# clinical_vcells.drop(columns=drop_vcells, inplace=True)
# # fill na with 0
# cols_added = list(set(clinical_vcells) - set(clinical_data))
clinical_vcells[binary_vcells.columns] = clinical_vcells[binary_vcells.columns].fillna(value=0)
clinical_vcells

,patientID,sex,age,LDH,PFS_status,PFS_month,BOR,BRAF_mut,pre_MAPKi_treatment,source,...,TNF,TP53,UBE4B,USP29,USP42,USP7,VIM,YWHAH,YY1,ZDHHC9
0,CS_VT030,male,52,normal,1.0,6.5,PR,V600E,Chemotherapy,doi:10.1200/PO.16.00054,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,CS_VT006,female,53,elevated,0.0,44.6,SD,V600E,no,doi:10.1200/PO.16.00054,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,CS_VT014,male,63,normal,0.0,6.4,SD,V600E; R558Q,no,doi:10.1200/PO.16.00054,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,CS_VT010,male,41,NaN,1.0,8.0,CR,V600E,no,doi:10.1200/PO.16.00054,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,CS_VT011,male,41,elevated,1.0,9.9,PR,V600E,no,doi:10.1200/PO.16.00054,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,VS_Pat_66,male,71,NaN,1.0,1.5,PD,V600E,no,doi:10.1158/2159-8290.CD-13-0617,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
107,VS_Pat_70,female,60,NaN,1.0,7.2,PR,V600E,no,doi:10.1158/2159-8290.CD-13-0617,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
108,VS_Pat_73,male,51,NaN,1.0,6.2,PR,V600E,no,doi:10.1158/2159-8290.CD-13-0617,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
109,VS_Pat_74,male,58,NaN,1.0,7.0,PR,V600E,no,doi:10.1158/2159-8290.CD-13-0617,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [79]:
count_unique_value_per_class(clinical_vcells)

patientID
CS_VT030     1
CS_VT006     1
CS_VT014     1
CS_VT010     1
CS_VT011     1
            ..
VS_Pat_66    1
VS_Pat_70    1
VS_Pat_73    1
VS_Pat_74    1
VS_Pat_76    1
Name: count, Length: 111, dtype: int64

sex
male      63
female    48
Name: count, dtype: int64

age
51    7
41    5
54    5
61    5
47    4
62    4
55    4
44    4
58    4
48    4
52    3
53    3
63    3
59    3
68    3
60    3
40    3
49    3
43    3
57    3
50    3
45    2
36    2
73    2
72    2
56    2
23    2
66    2
75    1
38    1
69    1
83    1
31    1
70    1
67    1
46    1
77    1
21    1
26    1
29    1
76    1
25    1
74    1
64    1
71    1
42    1
Name: count, dtype: int64

LDH
NaN         49
normal      38
elevated    24
Name: count, dtype: int64

PFS_status
1.0    95
0.0    16
Name: count, dtype: int64

PFS_month
3.0    5
2.2    5
1.5    4
5.2    4
3.5    3
      ..
8.2    1
8.8    1
2.8    1
7.2    1
7.0    1
Name: count, Length: 69, dtype: int64

BOR
PR     56
SD     25
PD     17
CR      9
NaN

In [80]:
########## PATHWAYS P-VALUE ##########

# join clinical and pathways pvalues on patient id
clinical_pp = clinical_data.join(pathway_pvalues, on='patientID')
# # fill na with 0
# cols_added = list(set(clinical_pp) - set(clinical_data))
clinical_pp[pathway_pvalues.columns] = clinical_pp[pathway_pvalues.columns].fillna(value=0)
clinical_pp

,patientID,sex,age,LDH,PFS_status,PFS_month,BOR,BRAF_mut,pre_MAPKi_treatment,source,...,Allograft rejection,Graft-versus-host disease,Hypertrophic cardiomyopathy,Arrhythmogenic right ventricular cardiomyopathy,Dilated cardiomyopathy,Diabetic cardiomyopathy,Viral myocarditis,Lipid and atherosclerosis,Fluid shear stress and atherosclerosis,Melanoma Map (Curated)
0,CS_VT030,male,52,normal,1.0,6.5,PR,V600E,Chemotherapy,doi:10.1200/PO.16.00054,...,0.001355,0.000949,0.116164,0.000449,0.004221,0.337488,0.001415,0.201545,0.652893,0.443580
1,CS_VT006,female,53,elevated,0.0,44.6,SD,V600E,no,doi:10.1200/PO.16.00054,...,0.002168,0.001518,0.185877,0.000719,0.006753,0.003084,0.002265,0.006270,0.005180,0.030225
2,CS_VT014,male,63,normal,0.0,6.4,SD,V600E; R558Q,no,doi:10.1200/PO.16.00054,...,0.000813,0.000569,0.069695,0.954909,0.002532,0.001156,0.000849,0.002351,0.323523,0.271224
3,CS_VT010,male,41,NaN,1.0,8.0,CR,V600E,no,doi:10.1200/PO.16.00054,...,0.000813,0.000569,0.069695,0.000269,0.002532,0.001156,0.000849,0.002351,0.001942,0.271224
4,CS_VT011,male,41,elevated,1.0,9.9,PR,V600E,no,doi:10.1200/PO.16.00054,...,0.002981,0.002087,0.255601,0.783053,0.009287,0.276462,0.003114,0.156029,0.522543,0.167729
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,VS_Pat_66,male,71,NaN,1.0,1.5,PD,V600E,no,doi:10.1158/2159-8290.CD-13-0617,...,0.004879,0.003416,0.418332,0.001618,0.015199,0.237156,0.005097,0.014111,0.011658,0.000000
107,VS_Pat_70,female,60,NaN,1.0,7.2,PR,V600E,no,doi:10.1158/2159-8290.CD-13-0617,...,0.001897,0.001328,0.162638,0.000629,0.005909,0.697896,0.001982,0.182271,0.263321,0.398754
108,VS_Pat_73,male,51,NaN,1.0,6.2,PR,V600E,no,doi:10.1158/2159-8290.CD-13-0617,...,0.001355,0.000949,0.116164,0.000449,0.004221,0.001928,0.001415,0.003918,0.003237,0.231395
109,VS_Pat_74,male,58,NaN,1.0,7.0,PR,V600E,no,doi:10.1158/2159-8290.CD-13-0617,...,0.000542,0.000379,0.046462,1.000000,0.001688,0.000771,0.000566,0.001567,0.348934,0.301491


In [81]:
########## MELANOMA PATHWAY P-VALUE (KEGG) ##########

# add pvalues of melanoma pathway from kegg
clinical_pp_kegg = clinical_data.join(pathway_pvalues['Melanoma'], on='patientID')
clinical_pp_kegg['Melanoma'] = clinical_pp_kegg['Melanoma'].fillna(value=0)
clinical_pp_kegg

,patientID,sex,age,LDH,PFS_status,PFS_month,BOR,BRAF_mut,pre_MAPKi_treatment,source,cobimetinib,dabrafenib,trametinib,vemurafenib,Melanoma
0,CS_VT030,male,52,normal,1.0,6.5,PR,V600E,Chemotherapy,doi:10.1200/PO.16.00054,0.0,0.0,0.0,1.0,1.000000
1,CS_VT006,female,53,elevated,0.0,44.6,SD,V600E,no,doi:10.1200/PO.16.00054,0.0,1.0,0.0,0.0,0.173074
2,CS_VT014,male,63,normal,0.0,6.4,SD,V600E; R558Q,no,doi:10.1200/PO.16.00054,0.0,0.0,0.0,1.0,0.213052
3,CS_VT010,male,41,NaN,1.0,8.0,CR,V600E,no,doi:10.1200/PO.16.00054,0.0,0.0,0.0,1.0,0.476281
4,CS_VT011,male,41,elevated,1.0,9.9,PR,V600E,no,doi:10.1200/PO.16.00054,0.0,0.0,0.0,1.0,0.356818
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,VS_Pat_66,male,71,NaN,1.0,1.5,PD,V600E,no,doi:10.1158/2159-8290.CD-13-0617,0.0,0.0,0.0,1.0,0.137216
107,VS_Pat_70,female,60,NaN,1.0,7.2,PR,V600E,no,doi:10.1158/2159-8290.CD-13-0617,0.0,0.0,0.0,1.0,0.399332
108,VS_Pat_73,male,51,NaN,1.0,6.2,PR,V600E,no,doi:10.1158/2159-8290.CD-13-0617,0.0,0.0,0.0,1.0,0.430442
109,VS_Pat_74,male,58,NaN,1.0,7.0,PR,V600E,no,doi:10.1158/2159-8290.CD-13-0617,0.0,0.0,0.0,1.0,0.227374


In [82]:
########## MELANOMA PATHWAY P-VALUE (VCELLS) ##########

# add pvalues of melanoma pathway from vcells
clinical_pp_vcells = clinical_data.join(pathway_pvalues['Melanoma Map (Curated)'], on='patientID')
clinical_pp_vcells['Melanoma Map (Curated)'] = clinical_pp_vcells['Melanoma Map (Curated)'].fillna(value=0)
clinical_pp_vcells

,patientID,sex,age,LDH,PFS_status,PFS_month,BOR,BRAF_mut,pre_MAPKi_treatment,source,cobimetinib,dabrafenib,trametinib,vemurafenib,Melanoma Map (Curated)
0,CS_VT030,male,52,normal,1.0,6.5,PR,V600E,Chemotherapy,doi:10.1200/PO.16.00054,0.0,0.0,0.0,1.0,0.443580
1,CS_VT006,female,53,elevated,0.0,44.6,SD,V600E,no,doi:10.1200/PO.16.00054,0.0,1.0,0.0,0.0,0.030225
2,CS_VT014,male,63,normal,0.0,6.4,SD,V600E; R558Q,no,doi:10.1200/PO.16.00054,0.0,0.0,0.0,1.0,0.271224
3,CS_VT010,male,41,NaN,1.0,8.0,CR,V600E,no,doi:10.1200/PO.16.00054,0.0,0.0,0.0,1.0,0.271224
4,CS_VT011,male,41,elevated,1.0,9.9,PR,V600E,no,doi:10.1200/PO.16.00054,0.0,0.0,0.0,1.0,0.167729
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,VS_Pat_66,male,71,NaN,1.0,1.5,PD,V600E,no,doi:10.1158/2159-8290.CD-13-0617,0.0,0.0,0.0,1.0,0.000000
107,VS_Pat_70,female,60,NaN,1.0,7.2,PR,V600E,no,doi:10.1158/2159-8290.CD-13-0617,0.0,0.0,0.0,1.0,0.398754
108,VS_Pat_73,male,51,NaN,1.0,6.2,PR,V600E,no,doi:10.1158/2159-8290.CD-13-0617,0.0,0.0,0.0,1.0,0.231395
109,VS_Pat_74,male,58,NaN,1.0,7.0,PR,V600E,no,doi:10.1158/2159-8290.CD-13-0617,0.0,0.0,0.0,1.0,0.301491


In [83]:

########## COMBINED DATASET ##########

## Combine all additional features
# common genes between kegg and vcells network
common_genes = np.intersect1d(binary_kegg.columns, binary_vcells.columns)
# add kegg network data
combined_c_k = clinical_data.join(binary_kegg, on='patientID')
# add vcells network data ( dropped common genes )
combined_c_k_v = combined_c_k.join(binary_vcells.drop(columns=common_genes), on='patientID')
# add pathway pvalues
combined_c_k_v_p = combined_c_k_v.join(pathway_pvalues, on='patientID')
# # Drop empty columns
# #combined_c_k_v_p.dropna(how='all', axis=1, inplace=True)
# drop_cols = threshold_level(combined_c_k_v_p, combined_c_k_v_p.drop(columns=clinical_data.columns).columns)
# combined_c_k_v_p.drop(columns=drop_cols, inplace=True)
# fill na with 0
cols_added = list(set(combined_c_k_v_p) - set(clinical_data))
combined_c_k_v_p[cols_added] = combined_c_k_v_p[cols_added].fillna(value=0)
combined_c_k_v_p

,patientID,sex,age,LDH,PFS_status,PFS_month,BOR,BRAF_mut,pre_MAPKi_treatment,source,...,Allograft rejection,Graft-versus-host disease,Hypertrophic cardiomyopathy,Arrhythmogenic right ventricular cardiomyopathy,Dilated cardiomyopathy,Diabetic cardiomyopathy,Viral myocarditis,Lipid and atherosclerosis,Fluid shear stress and atherosclerosis,Melanoma Map (Curated)
0,CS_VT030,male,52,normal,1.0,6.5,PR,V600E,Chemotherapy,doi:10.1200/PO.16.00054,...,0.001355,0.000949,0.116164,0.000449,0.004221,0.337488,0.001415,0.201545,0.652893,0.443580
1,CS_VT006,female,53,elevated,0.0,44.6,SD,V600E,no,doi:10.1200/PO.16.00054,...,0.002168,0.001518,0.185877,0.000719,0.006753,0.003084,0.002265,0.006270,0.005180,0.030225
2,CS_VT014,male,63,normal,0.0,6.4,SD,V600E; R558Q,no,doi:10.1200/PO.16.00054,...,0.000813,0.000569,0.069695,0.954909,0.002532,0.001156,0.000849,0.002351,0.323523,0.271224
3,CS_VT010,male,41,NaN,1.0,8.0,CR,V600E,no,doi:10.1200/PO.16.00054,...,0.000813,0.000569,0.069695,0.000269,0.002532,0.001156,0.000849,0.002351,0.001942,0.271224
4,CS_VT011,male,41,elevated,1.0,9.9,PR,V600E,no,doi:10.1200/PO.16.00054,...,0.002981,0.002087,0.255601,0.783053,0.009287,0.276462,0.003114,0.156029,0.522543,0.167729
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,VS_Pat_66,male,71,NaN,1.0,1.5,PD,V600E,no,doi:10.1158/2159-8290.CD-13-0617,...,0.004879,0.003416,0.418332,0.001618,0.015199,0.237156,0.005097,0.014111,0.011658,0.000000
107,VS_Pat_70,female,60,NaN,1.0,7.2,PR,V600E,no,doi:10.1158/2159-8290.CD-13-0617,...,0.001897,0.001328,0.162638,0.000629,0.005909,0.697896,0.001982,0.182271,0.263321,0.398754
108,VS_Pat_73,male,51,NaN,1.0,6.2,PR,V600E,no,doi:10.1158/2159-8290.CD-13-0617,...,0.001355,0.000949,0.116164,0.000449,0.004221,0.001928,0.001415,0.003918,0.003237,0.231395
109,VS_Pat_74,male,58,NaN,1.0,7.0,PR,V600E,no,doi:10.1158/2159-8290.CD-13-0617,...,0.000542,0.000379,0.046462,1.000000,0.001688,0.000771,0.000566,0.001567,0.348934,0.301491


In [84]:
count_unique_value_per_class(combined_c_k_v_p)

patientID
CS_VT030     1
CS_VT006     1
CS_VT014     1
CS_VT010     1
CS_VT011     1
            ..
VS_Pat_66    1
VS_Pat_70    1
VS_Pat_73    1
VS_Pat_74    1
VS_Pat_76    1
Name: count, Length: 111, dtype: int64

sex
male      63
female    48
Name: count, dtype: int64

age
51    7
41    5
54    5
61    5
47    4
62    4
55    4
44    4
58    4
48    4
52    3
53    3
63    3
59    3
68    3
60    3
40    3
49    3
43    3
57    3
50    3
45    2
36    2
73    2
72    2
56    2
23    2
66    2
75    1
38    1
69    1
83    1
31    1
70    1
67    1
46    1
77    1
21    1
26    1
29    1
76    1
25    1
74    1
64    1
71    1
42    1
Name: count, dtype: int64

LDH
NaN         49
normal      38
elevated    24
Name: count, dtype: int64

PFS_status
1.0    95
0.0    16
Name: count, dtype: int64

PFS_month
3.0    5
2.2    5
1.5    4
5.2    4
3.5    3
      ..
8.2    1
8.8    1
2.8    1
7.2    1
7.0    1
Name: count, Length: 69, dtype: int64

BOR
PR     56
SD     25
PD     17
CR      9
NaN